In [2]:
import pandas as pd
from pathlib import Path

file_path = Path("../data/raw/FreshBasket_Loyalty_Churn_Dataset.xlsx")
print("Workbook exists:", file_path.exists())

Workbook exists: True


In [3]:
profile = pd.read_excel(file_path, sheet_name="fb Customer Profile", header=1)
activity = pd.read_excel(file_path, sheet_name="fb Monthly Activity", header=1)
labels = pd.read_excel(file_path, sheet_name="fb Churn Label", header=1)

for name, df in {"Profile": profile, "Activity": activity, "Labels": labels}.items():
    print(f"{name}: {df.shape[0]} rows × {df.shape[1]} columns")

display(profile.head(3))
display(activity.head(3))
display(labels.head(3))

Profile: 2608 rows × 10 columns
Activity: 28073 rows × 12 columns
Labels: 2605 rows × 6 columns


,CUSTOMER_ID,SIGNUP_DATE,AGE,GENDER,CITY,MEMBERSHIP_TIER,MARKETING_OPT_IN,PREFERRED_STORE_ID,PREFERRED_CATEGORY,HOME_STORE_PRICE_TIER
0,500001,2022-08-17,36.0,F,Madison,Silver,1,140,Frozen Foods,Value
1,500002,2023-06-24,37.0,F,Spokane,Silver,0,119,Beverages,Mainstream
2,500003,2021-03-31,53.0,NaN,Tulsa,Gold,0,119,Frozen Foods,Mainstream


,CUSTOMER_ID,MONTH,TRANSACTIONS,TOTAL_SPEND,AVG_BASKET_VALUE,DISTINCT_CATEGORIES,PROMO_TXN_PCT,APP_SESSIONS,EMAILS_OPENED,COUPONS_REDEEMED,SUPPORT_TICKETS,COMPLAINT_FLAG
0,501855,2023-03-01,5,150.01,30.00,4,0.04,3,0,1,0,0
1,501803,2024-04-01,12,325.77,27.15,3,0.12,1,2,0,0,0
2,501889,2024-05-01,4,165.54,41.38,4,0.60,4,1,0,0,0


,CUSTOMER_ID,OBSERVATION_END_DATE,LAST_PURCHASE_DATE,MONTHS_OBSERVED,INSUFFICIENT_HISTORY_FLAG,CHURNED
0,500001,2024-06-30,2024-06-01,18,0,0
1,500002,2024-06-30,2024-06-01,12,0,0
2,500003,2024-06-30,2024-06-01,18,0,0


In [54]:
print("Churn label counts:")
print(labels["CHURNED"].value_counts())

print("\nExact duplicate rows:")
print({"profile": profile.duplicated().sum(),"activity": activity.duplicated().sum(),
    "labels": labels.duplicated().sum()
})


Churn label counts:
CHURNED
0    1784
1     821
Name: count, dtype: int64

Exact duplicate rows:
{'profile': np.int64(8), 'activity': np.int64(58), 'labels': np.int64(5)}


In [5]:
profile_clean = profile.drop_duplicates().copy()
activity_clean = activity.drop_duplicates().copy()
labels_clean = labels.drop_duplicates().copy()

print("Profile:", profile.shape, "->", profile_clean.shape)
print("Activity:", activity.shape, "->", activity_clean.shape)
print("Labels:", labels.shape, "->", labels_clean.shape)

Profile: (2608, 10) -> (2600, 10)
Activity: (28073, 12) -> (28015, 12)
Labels: (2605, 6) -> (2600, 6)


In [7]:
activity_dupes = activity_clean[
    activity_clean.duplicated(["CUSTOMER_ID", "MONTH"], keep=False)
].sort_values(["CUSTOMER_ID", "MONTH"])

print(activity_dupes)

conflict_keys = [
    (501780, pd.Timestamp("2024-05-01")),
    (502065, pd.Timestamp("2023-04-01"))
]


       CUSTOMER_ID      MONTH  TRANSACTIONS  TOTAL_SPEND  AVG_BASKET_VALUE  \
14414       501780 2024-05-01             3       116.18             38.73   
16384       501780 2024-05-01             3      -116.18             38.73   
15432       502065 2023-04-01             3       -70.32             23.44   
26567       502065 2023-04-01             3        70.32             23.44   

       DISTINCT_CATEGORIES  PROMO_TXN_PCT  APP_SESSIONS  EMAILS_OPENED  \
14414                    2           0.07             6              1   
16384                    2           0.07             6              1   
15432                    2           0.22             0              2   
26567                    2           0.22             0              2   

       COUPONS_REDEEMED  SUPPORT_TICKETS  COMPLAINT_FLAG  
14414                 0                0               0  
16384                 0                0               0  
15432                 1                0               0  
26

In [ ]:
activity_model = activity_clean.copy()

conflict_pairs = pd.MultiIndex.from_frame(
    activity_model[["CUSTOMER_ID", "MONTH"]]
).isin(conflict_keys)
activity_model = activity_model[
    ~(conflict_pairs & (activity_model["TOTAL_SPEND"] < 0))
].copy()
print("after conflict handling:",activity_model.duplicated(["CUSTOMER_ID", "MONTH"]).sum())

In [9]:
print("Missing values in cleaned tables:")
for name, df in {"Profile": profile_clean, "Activity": activity_model, "Labels": labels_clean}.items():
    print(f"\n{name}")
    print(df.isna().sum()[df.isna().sum() > 0])

negative_spend = activity_model["TOTAL_SPEND"] < 0

zero_spend_with_transactions = (
    (activity_model["TOTAL_SPEND"] == 0) &
    (activity_model["TRANSACTIONS"] > 0)
)

invalid_age = profile_clean["AGE"].notna() & ~profile_clean["AGE"].between(18, 100)
invalid_promo = ~activity_model["PROMO_TXN_PCT"].between(0, 1)

print("\nQuality-check counts:")
print("Negative-spend rows:", int(negative_spend.sum()))
print("Zero spend with positive transactions:", int(zero_spend_with_transactions.sum()))
print("Age outside 18–100:", int(invalid_age.sum()))
print("Promotion percentage outside 0–1:", int(invalid_promo.sum()))

Missing values in cleaned tables:

Profile
AGE                       78
GENDER                    52
HOME_STORE_PRICE_TIER    130
dtype: int64

Activity
Series([], dtype: int64)

Labels
LAST_PURCHASE_DATE    120
dtype: int64

Quality-check counts:
Negative-spend rows: 22
Zero spend with positive transactions: 20
Age outside 18–100: 0
Promotion percentage outside 0–1: 0


In [ ]:
# Compare recorded spend with transactions × average basket value.
activity_model["EXPECTED_SPEND"] = (
    activity_model["TRANSACTIONS"] * activity_model["AVG_BASKET_VALUE"]
)
activity_model["SPEND_DIFF"] = (
    activity_model["TOTAL_SPEND"] - activity_model["EXPECTED_SPEND"]
).abs()

print("Rows where spend differs from expected spend by more than $1:",
      int((activity_model["SPEND_DIFF"] > 1).sum()))

print("\nCustomer ID coverage:")
profile_ids = set(profile_clean["CUSTOMER_ID"])
label_ids = set(labels_clean["CUSTOMER_ID"])
activity_ids = set(activity_model["CUSTOMER_ID"])

print("Unique profile customers:", len(profile_ids))
print("Unique label customers:", len(label_ids))
print("Unique activity customers:", len(activity_ids))
print("Profile customers without churn labels:", len(profile_ids - label_ids))
print("Profile customers without any activity:", len(profile_ids - activity_ids))


In [21]:
profile_model = profile_clean.copy()
text_columns = [
    "GENDER", "CITY", "MEMBERSHIP_TIER",
    "PREFERRED_CATEGORY", "HOME_STORE_PRICE_TIER"
]
for col in text_columns:
    profile_model[col] = profile_model[col].astype("string").str.strip()


profile_model["GENDER"] = profile_model["GENDER"].str.upper()
for col in ["CITY", "MEMBERSHIP_TIER", "PREFERRED_CATEGORY", "HOME_STORE_PRICE_TIER"]:
    profile_model[col] = profile_model[col].str.title()

profile_model["GENDER"] = profile_model["GENDER"].fillna("Unknown")
profile_model["HOME_STORE_PRICE_TIER"] = profile_model["HOME_STORE_PRICE_TIER"].fillna("Unknown")
profile_model["AGE"] = profile_model["AGE"].fillna(profile_model["AGE"].median())

print("Profile rows:", len(profile_model))
print("Remaining profile missing values:", profile_model.isna().sum())

Profile rows: 2600
Remaining profile missing values: CUSTOMER_ID              0
SIGNUP_DATE              0
AGE                      0
GENDER                   0
CITY                     0
MEMBERSHIP_TIER          0
MARKETING_OPT_IN         0
PREFERRED_STORE_ID       0
PREFERRED_CATEGORY       0
HOME_STORE_PRICE_TIER    0
dtype: int64


In [24]:
pre_churn_activity = activity_model[
    activity_model["MONTH"] < "2024-04-01"
].copy()

print("Pre-churn activity rows:", len(pre_churn_activity))
print("First month:", pre_churn_activity["MONTH"].min())
print("Last month:", pre_churn_activity["MONTH"].max())

Pre-churn activity rows: 23253
First month: 2023-01-01 00:00:00
Last month: 2024-03-01 00:00:00


In [ ]:
sum_columns = [
    "TRANSACTIONS", "TOTAL_SPEND", "APP_SESSIONS",
    "EMAILS_OPENED", "COUPONS_REDEEMED", "SUPPORT_TICKETS"
]
activity_features = (
    pre_churn_activity.groupby("CUSTOMER_ID")[sum_columns]
    .sum()
    .add_suffix("_PRE")
)

mean_columns = [
    "TRANSACTIONS", "TOTAL_SPEND", "AVG_BASKET_VALUE",
    "PROMO_TXN_PCT", "APP_SESSIONS", "EMAILS_OPENED", "SUPPORT_TICKETS"
]
activity_means = (
    pre_churn_activity.groupby("CUSTOMER_ID")[mean_columns]
    .mean()
    .add_suffix("_AVG_PRE")
)
activity_features = activity_features.join(activity_means)

active_months = (
    pre_churn_activity.assign(ACTIVE_MONTH=lambda df: df["TRANSACTIONS"] > 0)
    .groupby("CUSTOMER_ID")["ACTIVE_MONTH"]
    .sum()
    .rename("ACTIVE_MONTHS_PRE")
)
activity_features = activity_features.join(active_months)
   
recent_activity = pre_churn_activity[
    pre_churn_activity["MONTH"] >= "2024-01-01"
]
recent_features = recent_activity.groupby("CUSTOMER_ID").agg(
    RECENT_TRANSACTIONS=("TRANSACTIONS", "sum"),
    RECENT_SPEND=("TOTAL_SPEND", "sum"),
    RECENT_APP_SESSIONS=("APP_SESSIONS", "sum"),
    RECENT_SUPPORT_TICKETS=("SUPPORT_TICKETS", "sum")
)
activity_features = activity_features.join(recent_features)

print("Customers with pre-period activity:", len(activity_features))
print("Activity feature columns:", len(activity_features.columns))

Customers with pre-period activity: 2481
Activity feature columns: 18


In [35]:
features_df = activity_features.reset_index()

model_data = (
    profile_model
    .merge(features_df, on="CUSTOMER_ID", how="left", validate="one_to_one")
    .merge(
        labels_clean[["CUSTOMER_ID", "CHURNED", "INSUFFICIENT_HISTORY_FLAG"]],
        on="CUSTOMER_ID",
        how="left",
        validate="one_to_one"
    )
)

print("Modelling table shape:", model_data.shape)
print("Unique customers:", model_data["CUSTOMER_ID"].nunique())

print("\nInsufficient-history flag counts:")
print(model_data["INSUFFICIENT_HISTORY_FLAG"].value_counts(dropna=False))

print("\nChurn counts by history flag:")
print(pd.crosstab(model_data["INSUFFICIENT_HISTORY_FLAG"], model_data["CHURNED"]))

Modelling table shape: (2600, 30)
Unique customers: 2600

Insufficient-history flag counts:
INSUFFICIENT_HISTORY_FLAG
0    2227
1     373
Name: count, dtype: int64

Churn counts by history flag:
CHURNED                       0    1
INSUFFICIENT_HISTORY_FLAG           
0                          1565  662
1                           217  156


In [ ]:
model_eligible = model_data[
    model_data["INSUFFICIENT_HISTORY_FLAG"] == 0
].copy()

missing_pre = model_eligible["TRANSACTIONS_PRE"].isna()
print("no recorded of transaction activity:", int(missing_pre.sum()))

display(
    model_eligible.loc[
        missing_pre,
        ["CUSTOMER_ID", "CHURNED", "INSUFFICIENT_HISTORY_FLAG",
         "TRANSACTIONS_PRE", "TOTAL_SPEND_PRE"]
    ]
)
model_eligible = model_eligible.loc[~missing_pre].copy()

print("\nFinal eligible customers:", len(model_eligible))
print("Churn distribution:")
print(model_eligible["CHURNED"].value_counts())
print("Churn rate:", round(model_eligible["CHURNED"].mean() * 100, 2), "%")

no recorded of transaction activity: 2


,CUSTOMER_ID,CHURNED,INSUFFICIENT_HISTORY_FLAG,TRANSACTIONS_PRE,TOTAL_SPEND_PRE
1387,501388,0,0,NaN,NaN
2217,502218,0,0,NaN,NaN



Final eligible customers: 2225
Churn distribution:
CHURNED
0    1563
1     662
Name: count, dtype: int64
Churn rate: 29.75 %


In [56]:
# Final missing-value summary for follow-up before modelling
missing_features = model_eligible.isna().sum().sort_values(ascending=False)
print(missing_features[missing_features > 0].to_string() or "No missing values")

print("\nFinal dataset shape:", model_eligible.shape)
print("Unique customer IDs:", model_eligible["CUSTOMER_ID"].nunique())

RECENT_SUPPORT_TICKETS    449
RECENT_APP_SESSIONS       449
RECENT_SPEND              449
RECENT_TRANSACTIONS       449

Final dataset shape: (2225, 30)
Unique customer IDs: 2225


In [84]:

output_path = Path("../data/processed/model_eligible.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

model_eligible.to_csv(output_path, index=False)

print("Saved successfully to:", output_path)
print("Dataset shape:", model_eligible.shape)

Saved successfully to: ..\data\processed\model_eligible.csv
Dataset shape: (2225, 30)
